<img src="https://hilpisch.com/tpq_logo_bic.png"
width="20%" align="right">

<a href="https://colab.research.google.com/github/yhilpisch/algocolab/blob/main/notebooks/03_cloud_deployment_monitoring.ipynb"
target="_blank"><img
src="https://colab.research.google.com/assets/colab-badge.svg"
alt="Open In Colab"/></a>


# Algorithmic Trading with Python & Google Colab

## Session 3 — From Notebook to Trading System

### Auditable Replay, Monitoring, and Risk Guardrails

The Python Quants GmbH | https://tpq.io<br>
© Dr. Yves J. Hilpisch | https://hilpisch.com

This session turns the exact Session 2 inference contract into an auditable
historical paper-trading replay. It checks batch/replay parity, records every
event in SQLite, exercises operational guardrails, and reconciles the final
state.


## 1. Reconnect to a Completed Session 2 Run

Session 3 never reconstructs an architecture from memory and never substitutes
random weights. The selected Drive bundle must contain a checksummed Session 2
checkpoint with its feature order, scaler, topology, weights, and threshold.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'src').is_dir():
            if (candidate / 'data' / 'eod_data.csv').is_file():
                return candidate
    raise FileNotFoundError(
        'Could not locate the companion repository root.'
    )
if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/algocolab')
    if not PROJECT_ROOT.exists():
        subprocess.run(
            [
                'git', 'clone',
                'https://github.com/yhilpisch/algocolab.git',
                str(PROJECT_ROOT),
            ],
            check=True,
        )
    RUNS_ROOT = Path('/content/drive/MyDrive/algo/runs')
else:
    PROJECT_ROOT = find_project_root()
    local_drive_runs = Path(
        '/Users/yves/Google Drive/My Drive/algo/runs'
    )
    RUNS_ROOT = Path(
        os.environ.get('WEBINAR_RUNS_ROOT', local_drive_runs)
    )
pointer_path = RUNS_ROOT / 'active_run.json'
if not pointer_path.is_file():
    raise FileNotFoundError(
        f'Run pointer not found: {pointer_path}. '
        'Run Sessions 1 and 2 first.'
    )
pointer = json.loads(pointer_path.read_text(encoding='utf-8'))
RUN_ID = str(pointer.get('run_id', '')).strip()
if not RUN_ID:
    raise ValueError('The active run pointer has no run ID.')
manifest_path = RUNS_ROOT / RUN_ID / 'manifest.json'
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text())
else:
    manifest = {}
session_two_commit = manifest.get(
    'session_code_commits',
    {},
).get('2', manifest.get('code_commit'))
if IN_COLAB and session_two_commit:
    try:
        subprocess.run(
            [
                'git', '-C', str(PROJECT_ROOT),
                'checkout', '--detach', session_two_commit,
            ],
            check=True,
        )
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            f'Cannot check out Session 2 commit {session_two_commit}. '
            'Use a run whose code commit exists in the companion repository.'
        ) from error
PERSIST_RESULTS = os.environ.get(
    'WEBINAR_PERSIST_RESULTS',
    str(IN_COLAB),
).lower() in {'1', 'true', 'yes'}
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Run: {RUN_ID}')

In [ ]:
from src.artifacts import RunBundle
from src.models import load_model_checkpoint
from src.session3 import persist_session_three, run_session_three
bundle = RunBundle.open(
    RUNS_ROOT,
    RUN_ID,
    required_session=2,
)
model, contract = load_model_checkpoint(
    bundle.path / 'session_2/model.pt'
)
parameter_count = sum(
    parameter.numel() for parameter in model.parameters()
)
assert contract['run_id'] == bundle.run_id
print(contract['model_config'])
print(contract['feature_names'])
print(f"Threshold: {contract['threshold']:.2f}")
print(f"Model parameters: {parameter_count:,}")

## 2. Event Timing Before Infrastructure

For target bar `t`, the replay builds features only from prices through
`t-1`, sets the position at the start of the bar, realizes return `t`, deducts
cost for absolute position turnover, and then evaluates drawdown. A future
transport adapter must preserve this timing contract; this notebook
demonstrates the replay path directly.


The wealth update uses log returns and one-way turnover:

\[
\Delta q_t=|q_t-q_{t-1}|,\qquad
r_t^{\mathrm{net}}=q_t r_t-c\Delta q_t,\qquad
V_t=V_{t-1}\exp(r_t^{\mathrm{net}}).
\]

Here \(q_t\) is the position and \(c=0.5\) basis points is the one-way cost.
A direct reversal has two turnover units and costs 1 basis point.


## 3. Run the Historical Paper-Trading Replay

SQLite is the durable event ledger for the demonstration. It records ticks,
signals, orders, and portfolio states under the same run ID. This remains a
paper-trading simulation—not a broker-connected production service.


In [ ]:
temporary = TemporaryDirectory()
database_path = Path(temporary.name) / 'paper_trading.db'
results = run_session_three(
    bundle,
    PROJECT_ROOT / 'data' / 'eod_data.csv',
    database_path,
    max_drawdown=0.10,
)
results.reconciliation

## 4. Prove Batch/Stream Parity

The deployment path rebuilds each feature vector from the historical buffer,
one observation at a time. Those features, probabilities, and positions must
match the batch research path within floating-point tolerance.


In [ ]:
results.parity

In [ ]:
assert results.reconciliation['parity_passed']
assert results.reconciliation['bars_recorded'] == (
    results.reconciliation['bars_expected']
)
print('Parity release gate: PASS')

## 5. Inspect the Ledger and Risk Response

The 10% drawdown guardrail overrides the model, creates an explicit flattening
order, sets the system to `HALTED`, and prevents re-entry. State continues to be
recorded after the halt so the audit trail remains complete.


In [ ]:
import sqlite3
import pandas as pd
with sqlite3.connect(results.database_path) as connection:
    ledger_counts = pd.read_sql_query(
        """
        SELECT 'ticks' AS table_name, COUNT(*) AS rows FROM ticks
        UNION ALL
        SELECT 'signals', COUNT(*) FROM signals
        UNION ALL
        SELECT 'orders', COUNT(*) FROM orders
        UNION ALL
        SELECT 'portfolio_state', COUNT(*) FROM portfolio_state
        """,
        connection,
    )
ledger_counts

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')
telemetry = results.telemetry.copy()
telemetry['timestamp'] = telemetry['timestamp'].astype('datetime64[ns]')
axes = telemetry.plot(
    x='timestamp',
    y=['nav'],
    figsize=(10, 4),
    color=['#002D5A'],
    legend=False,
)
axes.set(title='Paper-trading NAV with risk halt', ylabel='NAV')
axes.grid(alpha=0.35)
plt.show()

## 6. Inject Failures Deliberately

A guardrail is credible only when an adverse input produces observable,
testable evidence. The suite injects a nine-day stale-data gap, an 18.18% price
jump, and a loss large enough to require a flattening order.


In [ ]:
failure_cases = pd.DataFrame(
    [
        {
            'test': 'stale_observation',
            'input': '9-day gap',
            'expected': 'reject',
        },
        {
            'test': 'price_jump',
            'input': '18.18% jump',
            'expected': 'reject',
        },
        {
            'test': 'drawdown_flatten',
            'input': '2% loss with 1% limit',
            'expected': 'flatten and halt',
        },
    ]
)
failure_cases

In [ ]:
results.failure_tests

In [ ]:
assert results.failure_tests['passed'].all()
assert results.reconciliation['final_position'] == 0
assert results.reconciliation['risk_flatten_orders'] == 1
print('Operational checks: PASS')

## 7. Reconcile and Persist

The reconciliation summarizes row counts, final position and NAV, maximum
drawdown, turnover, halt state, and parity. A participant run adds the SQLite
database and portable CSV/JSON evidence to the same Drive bundle.


In [ ]:
if PERSIST_RESULTS:
    code_commit = subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    persist_session_three(
        bundle,
        results,
        code_commit=code_commit,
    )
    print(f'Session 3 persisted: {bundle.path}')
else:
    print('Persistence disabled; set WEBINAR_PERSIST_RESULTS=true to enable.')

> **Production deepening beyond the live skeleton**
>
> - broker authentication, acknowledgements, rejects, and idempotent order IDs;
> - heartbeat supervision, latency budgets, reconnect and replay semantics;
> - database/broker reconciliation and restart recovery;
> - secrets management, container hardening, monitoring, and alert escalation;
> - slippage, partial fills, financing, exposure limits, and kill switches;
> - model/data drift policy, approval workflow, and rollback.
>
> Passing this replay validates the educational contract. It is not production
> certification and does not authorize live trading.


## Series Takeaway

The three sessions form one traceable chain: test a narrow predictability
hypothesis, evaluate a non-linear alternative without contaminating the test
set, and deploy the exact frozen decision rule into an auditable paper-trading
replay. Negative economic results remain part of the evidence.


---

<img src="https://hilpisch.com/tpq_logo_bic.png"
width="20%" align="right">
